# Intraday Setup + 620 Cross Entry

This notebook detects intraday setup zones (base/rectangle) and marks entries when the 6/20 EMA cross appears.

Logic aligned with `docs/setup-detector.md`:
- Intraday setup: no 6/20 cross during setup, range stays contained for a minimum bar window.
- Entry: first 6/20 cross after setup completion (within a configurable wait window).

You can tune parameters in the config cell.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

In [22]:
# -----------------------------
# Config
# -----------------------------
TICKER = "SATS"
INTERVAL = "5m"       # try: 1m, 2m, 5m, 15m

# Data scope mode:
# - USE_SPECIFIC_DAY=True  -> use SPECIFIC_DAY only
# - USE_SPECIFIC_DAY=False -> use PERIOD window
USE_SPECIFIC_DAY = True
SPECIFIC_DAY = "2026-02-12"   # YYYY-MM-DD (exchange day)
EXCHANGE_TZ = "America/New_York"

PERIOD = "10d"        # used only when USE_SPECIFIC_DAY=False
INCLUDE_PREMARKET = True  # passed to yfinance as prepost

EMA_FAST = 6
EMA_SLOW = 20

MIN_BASE_BARS = 15
MAX_BASE_BARS = 40
MAX_RANGE_PCT = 0.018  # setup rectangle height cap
MAX_WAIT_FOR_CROSS = 20
MAX_WAIT_FOR_CLOSE_EMA20_CROSS = 20

# Compression rules (soft-scoring base definition)
EFFICIENCY_MAX = 0.65         # higher = allow more directional but still controlled
EMA_GAP_CAP_PCT = 0.0045      # relaxed EMA6-EMA20 normalized gap cap
EMA_GAP_NEG_FRAC_MIN = 0.55   # relaxed fraction of decreasing EMA-gap steps
DRIFT_CAP_PCT = 0.020         # relaxed max net move across base window
MIN_COMPRESSION_RULES = 3     # require at least N of 4 blocks (price/ema/slope/trend-significance)
COMPRESS_LOOKBACK_BARS = 7    # compression confirmation window

# Trend significance checks
UPTREND_REQUIRE_NEW_HIGH = True
DOWNTREND_REQUIRE_NEW_LOW = True
REQUIRE_TREND_DECELERATION = True  # in trend, pushes should become less significant
HARD_DOWNTREND_DECEL_GATE = True   # if downtrend, must decelerate in last COMPRESS_LOOKBACK_BARS

# Overlap merge settings (remove duplicate/nested rectangles)
MERGE_OVERLAPPING_ZONES = True
MERGE_MIN_PRICE_OVERLAP_RATIO = 0.35
MERGE_MAX_TIME_GAP_BARS = 1

BASE_COLOR = "rgba(59,130,246,0.18)"   # blue
BULL_COLOR = "#22c55e"
BEAR_COLOR = "#ef4444"

In [23]:
# -----------------------------
# Load intraday data
# -----------------------------
if USE_SPECIFIC_DAY:
    day_start = pd.Timestamp(SPECIFIC_DAY).tz_localize(EXCHANGE_TZ)
    day_end = day_start + pd.Timedelta(days=1)

    df = yf.download(
        TICKER,
        start=day_start,
        end=day_end,
        interval=INTERVAL,
#         prepost=INCLUDE_PREMARKET,
        auto_adjust=False,
        progress=False,
    )
else:
    df = yf.download(
        TICKER,
        period=PERIOD,
        interval=INTERVAL,
        prepost=INCLUDE_PREMARKET,
        auto_adjust=False,
        progress=False,
    )

if df.empty:
    raise ValueError("No data returned from yfinance. Try another ticker/interval/day or run later.")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [c[0].lower() for c in df.columns]
else:
    df.columns = [c.lower() for c in df.columns]

required_cols = {"open", "high", "low", "close"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for c in ["open", "high", "low", "close", "volume"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["open", "high", "low", "close"]).copy()

# Normalize all timestamps to US market time
if getattr(df.index, "tz", None) is None:
    # yfinance intraday is usually tz-aware; this is a safety fallback
    df.index = df.index.tz_localize("UTC").tz_convert(EXCHANGE_TZ)
else:
    df.index = df.index.tz_convert(EXCHANGE_TZ)

df["ema6"] = df["close"].ewm(span=EMA_FAST, adjust=False).mean()
df["ema20"] = df["close"].ewm(span=EMA_SLOW, adjust=False).mean()

if USE_SPECIFIC_DAY:
    print(f"Loaded {TICKER} {INTERVAL} for {SPECIFIC_DAY} | premarket={INCLUDE_PREMARKET} | tz={EXCHANGE_TZ} | rows={len(df)}")
else:
    print(f"Loaded {TICKER} {INTERVAL} for period={PERIOD} | premarket={INCLUDE_PREMARKET} | tz={EXCHANGE_TZ} | rows={len(df)}")

df.tail(3)

Loaded SATS 5m for 2026-02-12 | premarket=True | tz=America/New_York | rows=78


,adj close,close,high,low,open,volume,ema6,ema20
Datetime,,,,,,,,
2026-02-12 15:45:00-05:00,109.139999,109.139999,109.459999,109.084999,109.345001,39182,109.378722,109.653548
2026-02-12 15:50:00-05:00,110.445000,110.445000,110.519997,109.165001,109.165001,83147,109.683373,109.728925
2026-02-12 15:55:00-05:00,110.254997,110.254997,110.699997,110.070000,110.445000,279895,109.846694,109.779027


In [24]:
def classify_crosses(data: pd.DataFrame) -> pd.Series:
    """Return a series: +1 bullish cross, -1 bearish cross, 0 none."""
    diff = data["ema6"] - data["ema20"]
    prev = diff.shift(1)

    out = pd.Series(0, index=data.index, dtype=int)
    out[(prev <= 0) & (diff > 0)] = 1
    out[(prev >= 0) & (diff < 0)] = -1
    return out


def _lr_slope(values: np.ndarray) -> float:
    y = np.asarray(values, dtype=float)
    if y.size < 2 or not np.isfinite(y).all():
        return np.nan
    x = np.arange(y.size, dtype=float)
    return float(np.polyfit(x, y, 1)[0])


def _frac_negative(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=float)
    if x.size == 0:
        return 0.0
    return float(np.mean(x < 0))


def _interval_timedelta(index: pd.Index) -> pd.Timedelta:
    if len(index) < 2:
        return pd.Timedelta(minutes=5)
    diffs = pd.Series(index).diff().dropna()
    if diffs.empty:
        return pd.Timedelta(minutes=5)
    return pd.to_timedelta(diffs.median())


def _price_overlap_ratio(row_a: pd.Series, row_b: pd.Series) -> float:
    a0, a1 = float(row_a["base_low"]), float(row_a["base_high"])
    b0, b1 = float(row_b["base_low"]), float(row_b["base_high"])
    inter = max(0.0, min(a1, b1) - max(a0, b0))
    h_a = max(a1 - a0, 1e-9)
    h_b = max(b1 - b0, 1e-9)
    return inter / min(h_a, h_b)


def merge_overlapping_setups(
    setups: pd.DataFrame,
    bar_delta: pd.Timedelta,
    min_price_overlap_ratio: float = 0.35,
    max_time_gap_bars: int = 1,
) -> pd.DataFrame:
    if setups.empty:
        return setups.copy()

    s = setups.sort_values(["setup_start", "setup_end"]).reset_index(drop=True).copy()
    merged_rows = []
    cur = s.iloc[0].copy()

    for i in range(1, len(s)):
        nxt = s.iloc[i].copy()

        max_gap = bar_delta * max_time_gap_bars
        time_overlaps_or_touches = nxt["setup_start"] <= (cur["setup_end"] + max_gap)
        price_overlap_ok = _price_overlap_ratio(cur, nxt) >= min_price_overlap_ratio

        if time_overlaps_or_touches and price_overlap_ok:
            cur["setup_start"] = min(cur["setup_start"], nxt["setup_start"])
            cur["setup_end"] = max(cur["setup_end"], nxt["setup_end"])
            cur["base_low"] = min(float(cur["base_low"]), float(nxt["base_low"]))
            cur["base_high"] = max(float(cur["base_high"]), float(nxt["base_high"]))
            cur["bars_in_setup"] = int(max(cur.get("bars_in_setup", 0), nxt.get("bars_in_setup", 0)))

            # Keep earliest setup alert timestamp when zones merge
            if "alert_time" in cur and "alert_time" in nxt:
                if nxt["alert_time"] < cur["alert_time"]:
                    cur["alert_time"] = nxt["alert_time"]
                    if "alert_price" in nxt:
                        cur["alert_price"] = nxt["alert_price"]

            # Keep stronger-scoring zone metadata if available
            cur_score = int(cur.get("pass_count", 0))
            nxt_score = int(nxt.get("pass_count", 0))
            if nxt_score > cur_score:
                for k in [
                    "close20_cross_time",
                    "close20_cross_price",
                    "close20_cross_side",
                    "entry_time",
                    "entry_price",
                    "entry_side",
                    "pass_count",
                ]:
                    if k in nxt:
                        cur[k] = nxt[k]
        else:
            merged_rows.append(cur)
            cur = nxt

    merged_rows.append(cur)
    return pd.DataFrame(merged_rows).sort_values("setup_start").reset_index(drop=True)


def detect_intraday_setups_with_620_entry(
    data: pd.DataFrame,
    min_base_bars: int = 15,
    max_base_bars: int = 40,
    max_range_pct: float = 0.018,
    max_wait_for_cross: int = 20,
    max_wait_for_close_ema20_cross: int = 20,
    efficiency_max: float = 0.65,
    ema_gap_cap_pct: float = 0.0045,
    ema_gap_neg_frac_min: float = 0.55,
    drift_cap_pct: float = 0.020,
    min_compression_rules: int = 3,
    compress_lookback_bars: int = 7,
    uptrend_require_new_high: bool = True,
    downtrend_require_new_low: bool = True,
    require_trend_deceleration: bool = True,
    hard_downtrend_decel_gate: bool = True,
):
    """
    Detect setup rectangles (intraday base) and entry at first 6/20 cross after base.

    Setup requirements (scoring on rolling window):
    - no 6/20 cross inside setup window
    - price convergence (contained + non-expanding range, low efficiency)
    - EMA6/EMA20 gap compression
    - slope reduction (EMA slope magnitude and spread both decline)
    - if trend is up, evaluate whether new highs are meaningful (growing or decaying pushes)

    Entry:
    - first cross after setup end, within max_wait_for_cross bars
    """
    d = data.copy()
    d["cross"] = classify_crosses(d)

    n = len(d)
    setup_end_ok = np.zeros(n, dtype=bool)

    highs = d["high"].to_numpy()
    lows = d["low"].to_numpy()
    closes = d["close"].to_numpy()
    ema6 = d["ema6"].to_numpy()
    ema20 = d["ema20"].to_numpy()
    crosses = d["cross"].to_numpy()

    debug_metrics = {}

    # Candidate setup-end mask using fixed minimum window
    for end in range(min_base_bars - 1, n - 1):
        start = end - min_base_bars + 1

        win_high = float(np.max(highs[start : end + 1]))
        win_low = float(np.min(lows[start : end + 1]))
        width = win_high - win_low
        mid = max(float(closes[end]), 1e-9)
        width_pct = width / mid

        prev_width_pct = np.nan
        if start > 0:
            prev_high = float(np.max(highs[start - 1 : end]))
            prev_low = float(np.min(lows[start - 1 : end]))
            prev_width_pct = (prev_high - prev_low) / max(float(closes[end - 1]), 1e-9)

        eval_start = max(start, end - compress_lookback_bars + 1)

        close_win = closes[eval_start : end + 1]
        high_win = highs[eval_start : end + 1]
        path = float(np.sum(np.abs(np.diff(close_win))))
        efficiency = abs(float(close_win[-1]) - float(close_win[0])) / max(path, 1e-9)

        e6_win = ema6[eval_start : end + 1]
        e20_win = ema20[eval_start : end + 1]
        gap = np.abs(e6_win - e20_win) / np.maximum(close_win, 1e-9)
        gap_slope = _lr_slope(gap)
        gap_neg_frac = _frac_negative(np.diff(gap))

        slope6 = np.diff(e6_win)
        slope20 = np.diff(e20_win)
        slope_mag = np.abs(slope6) + np.abs(slope20)
        slope_spread = np.abs(slope6 - slope20)
        slope_mag_trend = _lr_slope(slope_mag)
        slope_spread_trend = _lr_slope(slope_spread)

        drift_pct = abs(float(close_win[-1]) - float(close_win[0])) / max(abs(float(close_win[0])), 1e-9)

        # Trend significance checks: new highs in uptrend / new lows in downtrend
        is_uptrend = bool(close_win[-1] > close_win[0] and e6_win[-1] >= e20_win[-1])
        is_downtrend = bool(close_win[-1] < close_win[0] and e6_win[-1] <= e20_win[-1])

        new_high_pushes = []
        run_max = -np.inf
        for h in high_win:
            if h > run_max:
                if np.isfinite(run_max):
                    new_high_pushes.append(float(h - run_max))
                run_max = h

        nh_count = len(new_high_pushes)
        if nh_count >= 2:
            nh_slope = _lr_slope(np.asarray(new_high_pushes, dtype=float))
            if nh_slope < 0:
                new_high_significance = "less"
            elif nh_slope > 0:
                new_high_significance = "more"
            else:
                new_high_significance = "flat"
        elif nh_count == 1:
            nh_slope = np.nan
            new_high_significance = "single"
        else:
            nh_slope = np.nan
            new_high_significance = "none"

        new_low_pushes = []
        run_min = np.inf
        for l in lows[eval_start : end + 1]:
            if l < run_min:
                if np.isfinite(run_min):
                    new_low_pushes.append(float(run_min - l))
                run_min = l

        nl_count = len(new_low_pushes)
        if nl_count >= 2:
            nl_slope = _lr_slope(np.asarray(new_low_pushes, dtype=float))
            if nl_slope < 0:
                new_low_significance = "less"
            elif nl_slope > 0:
                new_low_significance = "more"
            else:
                new_low_significance = "flat"
        elif nl_count == 1:
            nl_slope = np.nan
            new_low_significance = "single"
        else:
            nl_slope = np.nan
            new_low_significance = "none"

        if is_uptrend:
            if not uptrend_require_new_high:
                trend_significance_ok = True
            elif require_trend_deceleration:
                trend_significance_ok = new_high_significance in {"less", "flat"}
            else:
                trend_significance_ok = nh_count > 0
        elif is_downtrend:
            if not downtrend_require_new_low:
                trend_significance_ok = True
            elif require_trend_deceleration:
                trend_significance_ok = new_low_significance in {"less", "flat"}
            else:
                trend_significance_ok = nl_count > 0
        else:
            trend_significance_ok = True

        no_cross_inside = bool(np.all(crosses[start : end + 1] == 0))

        price_converging = (
            width_pct <= max_range_pct
            and (not np.isfinite(prev_width_pct) or width_pct <= prev_width_pct)
            and efficiency <= efficiency_max
        )

        ema_converging = (
            np.isfinite(gap_slope)
            and gap_slope < 0
            and float(gap[-1]) <= ema_gap_cap_pct
            and gap_neg_frac >= ema_gap_neg_frac_min
        )

        slope_reducing = (
            np.isfinite(slope_mag_trend)
            and np.isfinite(slope_spread_trend)
            and slope_mag_trend < 0
            and slope_spread_trend < 0
            and drift_pct <= drift_cap_pct
        )

        # Hard downtrend gate: in a downtrend window, lower-low behavior must be slowing
        # (or no fresh lower-low), evaluated on last COMPRESS_LOOKBACK_BARS bars.
        downtrend_decel_ok = (not is_downtrend) or (new_low_significance in {"less", "flat", "none"})

        pass_count = int(price_converging) + int(ema_converging) + int(slope_reducing) + int(trend_significance_ok)
        is_setup = no_cross_inside and pass_count >= min_compression_rules
        if hard_downtrend_decel_gate:
            is_setup = is_setup and downtrend_decel_ok

        setup_end_ok[end] = is_setup

        if is_setup:
            debug_metrics[end] = {
                "width_pct": width_pct,
                "efficiency": efficiency,
                "gap_end_pct": float(gap[-1]),
                "drift_pct": drift_pct,
                "price_converging": bool(price_converging),
                "ema_converging": bool(ema_converging),
                "slope_reducing": bool(slope_reducing),
                "uptrend": bool(is_uptrend),
                "downtrend": bool(is_downtrend),
                "new_high_count": int(nh_count),
                "new_high_significance": new_high_significance,
                "new_high_push_slope": float(nh_slope) if np.isfinite(nh_slope) else np.nan,
                "new_low_count": int(nl_count),
                "new_low_significance": new_low_significance,
                "new_low_push_slope": float(nl_slope) if np.isfinite(nl_slope) else np.nan,
                "trend_significance_ok": bool(trend_significance_ok),
                "downtrend_decel_ok": bool(downtrend_decel_ok),
                "pass_count": int(pass_count),
            }

    # Merge contiguous setup-end bars into setup segments
    segments = []
    i = 0
    while i < n:
        if not setup_end_ok[i]:
            i += 1
            continue

        j = i
        while j + 1 < n and setup_end_ok[j + 1]:
            j += 1

        seg_start = max(0, i - min_base_bars + 1)
        seg_end = j

        # Cap setup size from the right to avoid over-wide zones
        if seg_end - seg_start + 1 > max_base_bars:
            seg_start = seg_end - max_base_bars + 1

        base_high = float(np.max(highs[seg_start : seg_end + 1]))
        base_low = float(np.min(lows[seg_start : seg_end + 1]))

        # First close/EMA20 cross after setup alert
        close20_idx = None
        close20_side = None
        close20_scan_end = min(n - 1, seg_end + max_wait_for_close_ema20_cross)
        for k in range(seg_end + 1, close20_scan_end + 1):
            prev_close = closes[k - 1]
            curr_close = closes[k]
            prev_ema20 = ema20[k - 1]
            curr_ema20 = ema20[k]

            crossed_up = prev_close <= prev_ema20 and curr_close > curr_ema20
            crossed_down = prev_close >= prev_ema20 and curr_close < curr_ema20

            if crossed_up:
                close20_idx = k
                close20_side = "long"
                break
            if crossed_down:
                close20_idx = k
                close20_side = "short"
                break

        # First 620 cross after setup
        entry_idx = None
        entry_dir = None
        cross_scan_end = min(n - 1, seg_end + max_wait_for_cross)
        for k in range(seg_end + 1, cross_scan_end + 1):
            if crosses[k] == 1:
                entry_idx = k
                entry_dir = "long"
                break
            if crosses[k] == -1:
                entry_idx = k
                entry_dir = "short"
                break

        if entry_idx is not None:
            metrics = debug_metrics.get(seg_end, {})
            alert_time = d.index[seg_end]   # first alert at setup detection/completion
            alert_price = float(closes[seg_end])
            segments.append(
                {
                    "setup_start": d.index[seg_start],
                    "setup_end": d.index[seg_end],
                    "alert_time": alert_time,
                    "alert_price": alert_price,
                    "base_high": base_high,
                    "base_low": base_low,
                    "close20_cross_time": d.index[close20_idx] if close20_idx is not None else pd.NaT,
                    "close20_cross_price": float(closes[close20_idx]) if close20_idx is not None else np.nan,
                    "close20_cross_side": close20_side if close20_side is not None else "none",
                    "entry_time": d.index[entry_idx],
                    "entry_price": float(closes[entry_idx]),
                    "entry_side": entry_dir,
                    "bars_in_setup": int(seg_end - seg_start + 1),
                    "width_pct": round(float(metrics.get("width_pct", np.nan)), 5),
                    "efficiency": round(float(metrics.get("efficiency", np.nan)), 4),
                    "gap_end_pct": round(float(metrics.get("gap_end_pct", np.nan)), 5),
                    "drift_pct": round(float(metrics.get("drift_pct", np.nan)), 5),
                    "price_converging": bool(metrics.get("price_converging", False)),
                    "ema_converging": bool(metrics.get("ema_converging", False)),
                    "slope_reducing": bool(metrics.get("slope_reducing", False)),
                    "uptrend": bool(metrics.get("uptrend", False)),
                    "downtrend": bool(metrics.get("downtrend", False)),
                    "new_high_count": int(metrics.get("new_high_count", 0)),
                    "new_high_significance": metrics.get("new_high_significance", "none"),
                    "new_high_push_slope": float(metrics.get("new_high_push_slope", np.nan)),
                    "new_low_count": int(metrics.get("new_low_count", 0)),
                    "new_low_significance": metrics.get("new_low_significance", "none"),
                    "new_low_push_slope": float(metrics.get("new_low_push_slope", np.nan)),
                    "trend_significance_ok": bool(metrics.get("trend_significance_ok", True)),
                    "downtrend_decel_ok": bool(metrics.get("downtrend_decel_ok", True)),
                    "pass_count": int(metrics.get("pass_count", 0)),
                }
            )

        i = j + 1

    setups_df = pd.DataFrame(segments)
    return d, setups_df


work_df, setups_raw = detect_intraday_setups_with_620_entry(
    df,
    min_base_bars=MIN_BASE_BARS,
    max_base_bars=MAX_BASE_BARS,
    max_range_pct=MAX_RANGE_PCT,
    max_wait_for_cross=MAX_WAIT_FOR_CROSS,
    max_wait_for_close_ema20_cross=MAX_WAIT_FOR_CLOSE_EMA20_CROSS,
    efficiency_max=EFFICIENCY_MAX,
    ema_gap_cap_pct=EMA_GAP_CAP_PCT,
    ema_gap_neg_frac_min=EMA_GAP_NEG_FRAC_MIN,
    drift_cap_pct=DRIFT_CAP_PCT,
    min_compression_rules=MIN_COMPRESSION_RULES,
    compress_lookback_bars=COMPRESS_LOOKBACK_BARS,
    uptrend_require_new_high=UPTREND_REQUIRE_NEW_HIGH,
    downtrend_require_new_low=DOWNTREND_REQUIRE_NEW_LOW,
    require_trend_deceleration=REQUIRE_TREND_DECELERATION,
    hard_downtrend_decel_gate=HARD_DOWNTREND_DECEL_GATE,
)

if MERGE_OVERLAPPING_ZONES:
    bar_delta = _interval_timedelta(work_df.index)
    setups = merge_overlapping_setups(
        setups_raw,
        bar_delta=bar_delta,
        min_price_overlap_ratio=MERGE_MIN_PRICE_OVERLAP_RATIO,
        max_time_gap_bars=MERGE_MAX_TIME_GAP_BARS,
    )
else:
    setups = setups_raw.copy()

print(f"Detected raw setups: {len(setups_raw)} | merged setups: {len(setups)}")
setups.tail(10)

Detected raw setups: 4 | merged setups: 3


,setup_start,setup_end,alert_time,alert_price,base_high,base_low,close20_cross_time,close20_cross_price,close20_cross_side,entry_time,entry_price,entry_side,bars_in_setup,width_pct,efficiency,gap_end_pct,drift_pct,price_converging,ema_converging,slope_reducing,uptrend,downtrend,new_high_count,new_high_significance,new_high_push_slope,new_low_count,new_low_significance,new_low_push_slope,trend_significance_ok,downtrend_decel_ok,pass_count
0,2026-02-12 09:40:00-05:00,2026-02-12 11:20:00-05:00,2026-02-12 10:50:00-05:00,108.769997,110.653603,107.250000,2026-02-12 11:25:00-05:00,108.065002,short,2026-02-12 11:40:00-05:00,109.309998,long,18,0.03129,0.1299,0.00272,0.00197,False,True,True,False,True,1,single,NaN,3,less,-0.080002,True,True,4
1,2026-02-12 11:55:00-05:00,2026-02-12 13:20:00-05:00,2026-02-12 13:20:00-05:00,110.279999,111.559998,109.599998,2026-02-12 13:40:00-05:00,110.334999,long,2026-02-12 13:30:00-05:00,110.085503,short,18,0.01375,0.1087,0.00189,0.00181,True,True,False,False,False,0,none,NaN,2,less,-0.023994,True,True,3
2,2026-02-12 13:35:00-05:00,2026-02-12 14:45:00-05:00,2026-02-12 14:45:00-05:00,110.120003,110.410004,108.919998,2026-02-12 15:15:00-05:00,109.739998,short,2026-02-12 14:50:00-05:00,110.099998,long,15,0.01353,0.6040,0.00051,0.00599,True,True,False,False,False,5,more,0.026,0,none,NaN,True,True,3


In [25]:
fig = go.Figure()

# Candles
fig.add_trace(
    go.Candlestick(
        x=work_df.index,
        open=work_df["open"],
        high=work_df["high"],
        low=work_df["low"],
        close=work_df["close"],
        name="Price",
        increasing_line_color="#22c55e",
        decreasing_line_color="#ef4444",
        opacity=0.9,
    )
)

# EMA 6 / EMA 20
fig.add_trace(go.Scatter(x=work_df.index, y=work_df["ema6"], mode="lines", name="EMA 6", line=dict(color="#f59e0b", width=1.5)))
fig.add_trace(go.Scatter(x=work_df.index, y=work_df["ema20"], mode="lines", name="EMA 20", line=dict(color="#38bdf8", width=1.5)))

# Rectangles + setup alerts + entries
for _, r in setups.iterrows():
    fig.add_shape(
        type="rect",
        x0=r["setup_start"],
        x1=r["setup_end"],
        y0=r["base_low"],
        y1=r["base_high"],
        xref="x",
        yref="y",
        fillcolor=BASE_COLOR,
        line=dict(color="#3b82f6", width=1),
        layer="below",
    )

    # First alert: setup detected
    setup_mid = (float(r["base_low"]) + float(r["base_high"])) / 2.0
    fig.add_trace(
        go.Scatter(
            x=[r["alert_time"]],
            y=[setup_mid],
            mode="markers+text",
            marker=dict(size=10, symbol="diamond", color="#a78bfa", line=dict(width=1, color="#111827")),
            text=["SETUP"],
            textposition="top center",
            name="Setup Detected",
            showlegend=False,
        )
    )

    # Secondary alert: first close crossing EMA20 after setup alert
    if pd.notna(r["close20_cross_time"]):
        c20_side = r["close20_cross_side"]
        c20_color = BULL_COLOR if c20_side == "long" else BEAR_COLOR
        c20_symbol = "circle" if c20_side == "long" else "circle-open"

        fig.add_trace(
            go.Scatter(
                x=[r["close20_cross_time"]],
                y=[r["close20_cross_price"]],
                mode="markers+text",
                marker=dict(size=10, symbol=c20_symbol, color=c20_color, line=dict(width=1, color="#111827")),
                text=[f"CLOSE x EMA20 ({c20_side.upper()})"],
                textposition="top center" if c20_side == "long" else "bottom center",
                name="Close x EMA20",
                showlegend=False,
            )
        )

    side = r["entry_side"]
    color = BULL_COLOR if side == "long" else BEAR_COLOR
    marker = "triangle-up" if side == "long" else "triangle-down"

    fig.add_trace(
        go.Scatter(
            x=[r["entry_time"]],
            y=[r["entry_price"]],
            mode="markers+text",
            marker=dict(size=12, symbol=marker, color=color, line=dict(width=1, color="#111827")),
            text=[f"{side.upper()} 620"],
            textposition="top center" if side == "long" else "bottom center",
            name=f"{side.capitalize()} Entry",
            showlegend=False,
        )
    )

fig.update_layout(
    title=f"{TICKER} — Setup Alert + Close x EMA20 + 620 Entry",
    template="plotly_dark",
    height=820,
    xaxis_rangeslider_visible=False,
    legend=dict(orientation="h", y=1.02, x=0),
)

fig.show()

In [6]:
# Optional quick summary
if setups.empty:
    print("No setup+entry found with current parameters. Try loosening MAX_RANGE_PCT or increasing PERIOD.")
else:
    summary = setups.copy()
    summary["setup_height_pct"] = (summary["base_high"] - summary["base_low"]) / ((summary["base_high"] + summary["base_low"]) / 2.0)
    display_cols = [
        "setup_start",
        "setup_end",
        "alert_time",
        "alert_price",
        "close20_cross_time",
        "close20_cross_price",
        "close20_cross_side",
        "bars_in_setup",
        "base_low",
        "base_high",
        "entry_time",
        "entry_price",
        "entry_side",
        "setup_height_pct",
    ]
    display(summary[display_cols].tail(20))

,setup_start,setup_end,alert_time,alert_price,close20_cross_time,close20_cross_price,close20_cross_side,bars_in_setup,base_low,base_high,entry_time,entry_price,entry_side,setup_height_pct
0,2026-02-18 09:50:00-05:00,2026-02-18 11:20:00-05:00,2026-02-18 11:00:00-05:00,116.019997,2026-02-18 11:25:00-05:00,115.580002,short,15,114.830002,116.470001,2026-02-18 11:40:00-05:00,115.324997,short,0.014181
1,2026-02-18 13:25:00-05:00,2026-02-18 15:10:00-05:00,2026-02-18 14:35:00-05:00,113.709999,2026-02-18 15:30:00-05:00,113.919998,long,16,113.360001,115.089996,2026-02-18 15:45:00-05:00,114.660004,long,0.015146
